In [1]:
import numpy as np
import random

class SpiderMonkeyOptimization:
    def __init__(self, n_monkeys, max_iter, search_space, fitness_func):
        self.n_monkeys = n_monkeys  # Number of spider monkeys
        self.max_iter = max_iter    # Maximum number of iterations
        self.search_space = search_space  # Hyperparameter bounds
        self.fitness_func = fitness_func  # Function to optimize
        self.dimensions = len(search_space)  # Number of hyperparameters (dimensions)

    # Initialize the population with random values within the search space
    def initialize_population(self):
        population = []
        for _ in range(self.n_monkeys):
            monkey = [random.uniform(self.search_space[i][0], self.search_space[i][1]) for i in range(self.dimensions)]
            population.append(monkey)
        return population

    # Evaluate the fitness of the population
    def evaluate_population(self, population):
        fitness = []
        for monkey in population:
            fitness.append(self.fitness_func(monkey))
        return np.array(fitness)

    # Update monkey positions based on local and global leaders
    def update_monkey_positions(self, population, fitness, global_best, local_best, phase):
        new_population = []
        for i in range(len(population)):
            new_monkey = []
            for j in range(self.dimensions):
                if phase == 'local':
                    r = random.random()
                    new_val = population[i][j] + r * (local_best[j] - population[i][j])
                elif phase == 'global':
                    r = random.random()
                    new_val = population[i][j] + r * (global_best[j] - population[i][j])

                # Clamp values within the search space bounds
                new_val = np.clip(new_val, self.search_space[j][0], self.search_space[j][1])
                new_monkey.append(new_val)
            new_population.append(new_monkey)
        return new_population

    # Get the best solution (monkey) in the population
    def get_best(self, population, fitness):
        best_index = np.argmin(fitness)
        return population[best_index], fitness[best_index]

    # SMO optimization loop
    def optimize(self):
        population = self.initialize_population()
        fitness = self.evaluate_population(population)
        global_best, global_best_fitness = self.get_best(population, fitness)

        for iter in range(self.max_iter):
            # Local leader phase
            local_best, _ = self.get_best(population, fitness)
            population = self.update_monkey_positions(population, fitness, global_best, local_best, 'local')
            fitness = self.evaluate_population(population)
            
            # Update global leader
            new_global_best, new_global_best_fitness = self.get_best(population, fitness)
            if new_global_best_fitness < global_best_fitness:
                global_best, global_best_fitness = new_global_best, new_global_best_fitness

            # Global leader phase
            population = self.update_monkey_positions(population, fitness, global_best, local_best, 'global')
            fitness = self.evaluate_population(population)
            
            # Update global leader again
            new_global_best, new_global_best_fitness = self.get_best(population, fitness)
            if new_global_best_fitness < global_best_fitness:
                global_best, global_best_fitness = new_global_best, new_global_best_fitness

            print(f"Iteration {iter+1}/{self.max_iter} - Best Fitness: {global_best_fitness}")

        return global_best, global_best_fitness
